# Generate GLOBAL DBOF — Unified Pipeline (`generate-global-gen`)

A single CLI command (`generate-global-gen`) and a single config file
(`configs/global_unified.yaml`) replace the three legacy scripts
(`generate-global`, `generate-global-osn`, `generate-global-depth`).

The **`--pipeline`** argument selects the data source; **`--subset`** (or
`active_subsets` in the YAML) selects which property group to compute.

### Pipeline variants

| `--pipeline` | Data source | Notes |
|---|---|---|
| `SURF` | OSN kerchunk + S3 forcing | Core ocean vars from kerchunk; wind/ice from S3 |
| `OSN` | OSN kerchunk only | All surface + wind vars from kerchunk endpoints |
| `DEPTH` | S3 timestep stores (full depth) | 3D fields reduced to 2D via depth strategies |

### Surface subsets (SURF / OSN)

| `--subset` | Fields | Output zarr |
|---|---|---|
| `native_fields` | `Theta`, `Salt`, `Eta`, `U`, `V`, `W`, `oceTAUX`, `oceTAUY`, `SIarea` | `native_fields.zarr` |
| `frontal_structure` | `gradb2`, `gradsalt2`, `gradtheta2`, `gradeta2`, `gradrho2`, `turner_angle` | `frontal_structure.zarr` |
| `kinematic` | `relative_vorticity`, `strain_n/s`, `strain_mag`, `divergence`, `coriolis_f`, `rossby_number`, `okubo_weiss` | `kinematic.zarr` |
| `frontogenesis` | `frontogenesis_tendency`, `ug`, `vg`, `frontogenesis_geo`, `frontogenesis_ageo` | `frontogenesis.zarr` |

### Depth subsets (DEPTH)

| `--subset` | Fields (expanded with depth suffixes) | Output zarr |
|---|---|---|
| `stratification` | `N2` x depths + `mixed_layer_depth`, `ml_heat_content` | `stratification.zarr` |
| `vertical_shear` | `vertical_shear`, `Ri` x depths | `vertical_shear.zarr` |
| `mixing_parameters` | `Fr`, `Ro`, `Bu` x depths | `mixing_parameters.zarr` |
| `ertel_pv` | `ertel_pv`, `ertel_pv_vertical`, `ertel_pv_tilt` x depths | `ertel_pv.zarr` |
| `buoyancy_fluxes` | `uB`, `vB`, `wB` x depths | `buoyancy_fluxes.zarr` |
| `surface_wind` | `oceTAUX`, `oceTAUY`, `oceQnet`, `wind_stress_curl`, `ekman_pumping`, `u/v_ekman` | `surface_wind.zarr` |
| `energetics` | `KE` x depths | `energetics.zarr` |
| `frontal_structure` | gradient fields x `[sfc]` | `frontal_structure.zarr` |
| `kinematic` | vorticity/strain fields x depths | `kinematic.zarr` |
| `frontogenesis` | frontogenesis fields x `[sfc]` | `frontogenesis.zarr` |
| `native_fields` | `Theta`, `Salt`, `Eta`, `U`, `V`, `W` x `[sfc]` | `native_fields.zarr` |
| `icearea` | `SIarea` | `icearea.zarr` |

Depth suffixes: `_sfc` (surface), `_z25m` (25 m), `_mld` (mixed-layer depth), `_mld_mean` (ML mean).

---

### Output layout

All subsets share the same S3 directory via a shared `run_id`:
```
s3://{bucket}/{folder}/{run_id}/{date_prefix}/native_fields.zarr
s3://{bucket}/{folder}/{run_id}/{date_prefix}/kinematic.zarr
...
```

In [ ]:
import datetime

# Initial set up

In the future we will want to update this to use docker or conda for end users.

## Local machine setup

#### Build the project
- `pip install .`

The `generate-global-gen` entry point is registered in `pyproject.toml`:
```toml
generate-global-gen = "dbof.cli.generate_global_GEN:main"
```

#### Install aws cli (optional)
You will want to install this if you want to manually see the data stored in the s3 bucket.

Example for installing on linux:
- `sudo curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"`
- `sudo unzip awscliv2.zip`
- `sudo ./aws/install`

## Running in NRP Jupyterhub

This is for running this notebook on NRP Jupyterhub.

This block simply builds the project and installs dependencies not already present on NRP Jupyterhub.
You can safely ignore pip warnings.

For serious projects, users should use conda or docker but this notebook is meant to be very simple and user friendly.

In [ ]:
#Set True if running on NRP Jupyterhub
RUNNING_ON_NRP = False


if (RUNNING_ON_NRP):
    %pip install -e ../. --no-deps

    %pip install xgcm
    %pip install zarr
    %pip install boto3
    %pip install ujson
    %pip install scikit-fmm
    %pip install aiobotocore

# NOTE IF on NRP Jupyterhub, you will likely need to restart the kernel after running this block

## Set AWS credentials
If you are accessing or writing data to S3, you must set credentials.
NOTE: S3 is all that is supported currently.
This typically corresponds to an NRP S3 bucket.

windows:

- `$env:AWS_ACCESS_KEY_ID="..."`
- `$env:AWS_SECRET_ACCESS_KEY="..."`

unix:

- `export AWS_ACCESS_KEY_ID=...`
- `export AWS_SECRET_ACCESS_KEY=...`

# Dataset generation config (quick reference)

This job is fully controlled by `configs/global_unified.yaml`.  The config
defines **the pipeline variant**, **which dates to process**, and **which
property subsets to compute**.

### Pipeline selection
- `pipeline` in the YAML sets the default (`SURF`, `OSN`, or `DEPTH`).
- `--pipeline` on the CLI always takes precedence.

### Subset selection
- `active_subsets` in the YAML lists which subsets to compute.
- `--subset` on the CLI overrides with a single subset.
- Subset definitions (channel lists, dataset names, depth suffixes) are
  defined in code (`subset_definitions.py`), **not** in the YAML.

### Date iterations
- `data.date_iterations` lists ISO-format date strings.
- For DEPTH: each must match a transferred timestep store in S3.
- For SURF/OSN: each must fall within data availability windows.

### Depth suffixes (DEPTH only)
- Default: `[sfc, z25m, mld, mld_mean]`.
- Override in the YAML with `depth_suffixes: [sfc, mld]` (etc.).

### Output / logging
- `output.bucket`, `output.folder`: S3 location for dataset output.
- `run.run_id`: unique identifier for this session.
- `run.log_dir`: local directory where logs are written.

### Dask note for frontogenesis
The frontogenesis subset merges two large lazy lineages.  The compute function
mitigates this with a single `dask.compute()` call.  If you see scheduler
warnings, reduce `runtime.zarr_async_concurrency` in the config.

# A note about logs
The run logs will be stored locally on your machine in the directory you specify.
- `log_dir/run_id/`

Under the current logic, if you attempt to run the script and the specified
log output path already exists, the script will fail to run.  This is by design.
- `run_id` is also used for the zarr dataset path.  You likely do not want to
  send two different runs to the same dataset.
- You will likely not want to override your previous run logs.

**Exception:** when running multiple subsets with a shared `run_id` (the
intended usage here), the log directory is created once on the first subset
run and reused for subsequent subsets — each appending to the same
`generate_global_GEN.log` file.  This is expected behaviour.

To start a completely fresh session, generate a new `run_id` in the cell below.

# Generate a shared run_id for this session

Run this cell **once** at the start of a session.  Reuse `run_id` for every
subset you run below — this groups all output zarr files into the same S3
directory:
```
s3://{bucket}/{folder}/{run_id}/{date_prefix}/kinematic.zarr
s3://{bucket}/{folder}/{run_id}/{date_prefix}/frontogenesis.zarr
... etc.
```

In [ ]:
#run_id = f"global_{datetime.datetime.now(datetime.UTC).strftime('%Y%m%d_%H%M%S')}"
run_id = "global_test00"  # <-- update if different
print(f"Shared run_id for this session: {run_id}")
print(f"All subsets will write to: s3://dbof/properties/{run_id}/")
print()
print("Update configs/data_access/global_unified.yaml with this run_id when the run is complete.")

# Select pipeline variant

Set the pipeline to `SURF`, `OSN`, or `DEPTH`.  This can also be set in
`global_unified.yaml` (the `pipeline` key) — the CLI `--pipeline` flag
takes precedence.

In [ ]:
# Pipeline variant: SURF | OSN | DEPTH
pipeline = "DEPTH"
print(f"Pipeline: {pipeline}")

---
# Run subsets

Run the cells below **one at a time** (or comment out subsets you don't need).
All cells reuse the `run_id` and `pipeline` set above.

All Dask logs are warnings — do not be alarmed.

### Running multiple subsets in one call

You can also list multiple subsets in `active_subsets` in the config YAML
and omit `--subset` to process them all sequentially.

## Surface subsets (SURF / OSN)

These subsets are available when `pipeline` is `SURF` or `OSN`.

### `native_fields`
Raw model state variables: `Theta`, `Salt`, `Eta`, `U`, `V`, `W`, `oceTAUX`, `oceTAUY`, `SIarea`.

In [ ]:
!generate-global-gen \
    --config ../../configs/global_unified.yaml \
    --pipeline $pipeline \
    --subset native_fields \
    --run_id $run_id

### `frontal_structure`
`gradb2`, `gradsalt2`, `gradtheta2`, `gradeta2`, `gradrho2`, `turner_angle`.

In [ ]:
!generate-global-gen \
    --config ../../configs/global_unified.yaml \
    --pipeline $pipeline \
    --subset frontal_structure \
    --run_id $run_id

### `kinematic`
Velocity-derived scalar fields from a single Jacobian pass.

In [ ]:
!generate-global-gen \
    --config ../../configs/global_unified.yaml \
    --pipeline $pipeline \
    --subset kinematic \
    --run_id $run_id

### `frontogenesis`
`frontogenesis_tendency`, `ug`, `vg`, `frontogenesis_geo`, `frontogenesis_ageo`.

In [ ]:
!generate-global-gen \
    --config ../../configs/global_unified.yaml \
    --pipeline $pipeline \
    --subset frontogenesis \
    --run_id $run_id

## Depth subsets (DEPTH)

These subsets are available when `pipeline` is `DEPTH`.  All compute from
full-depth 3D fields and reduce to 2D surface output via depth strategies.

### `stratification`
MLD, N² at 4 depths, ML heat content.

In [ ]:
!generate-global-gen \
    --config ../../configs/global_unified.yaml \
    --pipeline $pipeline \
    --subset stratification \
    --run_id $run_id

### `vertical_shear`
Vertical shear and Richardson number at 4 depths.

In [ ]:
!generate-global-gen \
    --config ../../configs/global_unified.yaml \
    --pipeline $pipeline \
    --subset vertical_shear \
    --run_id $run_id

### `mixing_parameters`
Fr, Ro, Bu at 4 depths.

In [ ]:
!generate-global-gen \
    --config ../../configs/global_unified.yaml \
    --pipeline $pipeline \
    --subset mixing_parameters \
    --run_id $run_id

### `ertel_pv`
Ertel PV and its vertical/tilt components at 4 depths (12 channels).

In [ ]:
!generate-global-gen \
    --config ../../configs/global_unified.yaml \
    --pipeline $pipeline \
    --subset ertel_pv \
    --run_id $run_id

### `buoyancy_fluxes`
uB, vB, wB at 4 depths (12 channels).

In [ ]:
!generate-global-gen \
    --config ../../configs/global_unified.yaml \
    --pipeline $pipeline \
    --subset buoyancy_fluxes \
    --run_id $run_id

### `surface_wind`
Wind stress, Ekman pumping/transport, oceQnet.

In [ ]:
!generate-global-gen \
    --config ../../configs/global_unified.yaml \
    --pipeline $pipeline \
    --subset surface_wind \
    --run_id $run_id

### `energetics`
Kinetic energy at 4 depths.

In [ ]:
!generate-global-gen \
    --config ../../configs/global_unified.yaml \
    --pipeline $pipeline \
    --subset energetics \
    --run_id $run_id

### `icearea`
Sea-ice area fraction.

In [ ]:
!generate-global-gen \
    --config ../../configs/global_unified.yaml \
    --pipeline $pipeline \
    --subset icearea \
    --run_id $run_id

---
# S3 management

In [ ]:
# aws cli commands for listing or deleting data in the s3 bucket
# Replace {run_id} with your run_id value, or use the f-string version below.

# List all zarr stores for this run:
# aws --endpoint https://s3-west.nrp-nautilus.io s3 ls s3://dbof/properties/{run_id}/ --human-readable

# Delete an individual subset zarr (dry-run first):
# aws --endpoint https://s3-west.nrp-nautilus.io s3 rm s3://dbof/properties/{run_id}/20121109_120000/kinematic.zarr --recursive --dryrun

# Delete the entire run directory (dry-run first):
# aws --endpoint https://s3-west.nrp-nautilus.io s3 rm s3://dbof/properties/{run_id}/ --recursive --dryrun